
# Diversidade dos dados — Yahoo Finance 2023–2024 (v1.1 — download robusto)

Este notebook constrói, **do zero**, um banco diversificado de instâncias de seleção de portfólio para os próximos experimentos com VQE, QFIM e Transformer.

## Desenho do banco

A interpretação operacional adotada foi:

- universo de **60 ações**;
- seis grupos fixos e disjuntos de **10 ações**, cada grupo contendo uma ação de cada setor;
- para \(N=10\), gerar problemas pareados com:
  \[
  K\in\{2,3,4,5,6,7\};
  \]
- adicionar duas mudanças de dimensionalidade:
  \[
  (N,K)=(13,5),\qquad (N,K)=(20,5);
  \]
- gerar **1.000 instâncias para cada configuração**.

O banco padrão terá:

\[
6\times1.000 + 1.000 + 1.000 = 8.000
\]

instâncias.

## Princípio de pareamento

Para \(N=10\), uma mesma amostra de mercado gera seis Hamiltonianos, alterando apenas \(K\). Isso permite estudar diretamente:

\[
\text{mudança de }K
\longrightarrow
\text{mudança do Hamiltoniano}
\longrightarrow
\text{mudança posterior dos }\theta.
\]

Cada linha recebe um `paired_market_id`, permitindo comparar \(K=2,\ldots,7\) sob os mesmos retornos, covariância, ativos e aversão ao risco.

## O que este notebook faz agora

1. baixa preços ajustados de 2023;
2. constrói retornos diários;
3. gera diversidade por janelas e *moving-block bootstrap*;
4. estima retornos esperados e covariância regularizada;
5. constrói QUBO e Hamiltonianos de Ising;
6. calcula a solução clássica exata sob a restrição \(\sum_i x_i=K\);
7. salva dados tabulares e tensores prontos para o Transformer;
8. cria tabelas vazias para enriquecimento posterior com VQE, \(\theta\) e QFIM.

**A QFIM não é calculada neste notebook.** A prioridade aqui é produzir um banco limpo, pareado, reprodutível e pronto para receber esses rótulos depois.


## Correção v1.1

O download agora usa lotes pequenos, `threads=False`, repetição com espera,
fallback individual e cache parcial. A célula também mostra os erros internos
do `yfinance` quando o Yahoo retorna uma tabela vazia.


In [ ]:
# Execute esta célula uma única vez e REINICIE O KERNEL depois.
%pip install -U yfinance certifi curl_cffi pyarrow scikit-learn tqdm

In [ ]:

from __future__ import annotations

import hashlib
import json
import math
import platform
import random
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.covariance import LedoitWolf
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("yfinance:", yf.__version__)
print("Python:", sys.version)
print("Sistema:", platform.platform())


In [ ]:

# ============================================================
# CONFIGURAÇÃO CENTRAL
# ============================================================

START_DATE = "2023-01-01"
END_DATE = "2024-01-01"  # fim exclusivo: ano-calendário de 2023

RANDOM_SEED = 20260802
N_INSTANCES_PER_CONFIG = 1_000

N10_K_VALUES = [2, 3, 4, 5, 6, 7]
EXTRA_CONFIGS = [(13, 5), (20, 5)]

TRADING_DAYS = 252
WINDOW_LENGTHS = [63, 84, 126, 168, 210, 252]
BLOCK_LENGTHS = [5, 10, 15, 20]

RISK_AVERSION_MIN = 0.10
RISK_AVERSION_MAX = 0.90
PENALTY_MULTIPLIER = 2.50

MIN_PRICE_COVERAGE = 0.95
MAX_FORWARD_FILL_DAYS = 3

MAX_N = 20
COMPUTE_EXACT_LABELS = True
EXACT_SOLVER_BATCH_SIZE_N20 = 8

# Para validar rapidamente todo o pipeline antes da execução completa.
FAST_SMOKE_TEST = False
if FAST_SMOKE_TEST:
    N_INSTANCES_PER_CONFIG = 20
    COMPUTE_EXACT_LABELS = True

OUTPUT_DIR = Path("data_diversidade_yfinance_2023_2024")
RAW_DIR = OUTPUT_DIR / "raw"
TABLE_DIR = OUTPUT_DIR / "tables"
ARRAY_DIR = OUTPUT_DIR / "arrays"
REPORT_DIR = OUTPUT_DIR / "reports"
LABEL_DIR = OUTPUT_DIR / "future_labels"

for directory in [OUTPUT_DIR, RAW_DIR, TABLE_DIR, ARRAY_DIR, REPORT_DIR, LABEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED_TOTAL = (
    len(N10_K_VALUES) * N_INSTANCES_PER_CONFIG
    + len(EXTRA_CONFIGS) * N_INSTANCES_PER_CONFIG
)

print("Instâncias por configuração:", N_INSTANCES_PER_CONFIG)
print("Total esperado:", EXPECTED_TOTAL)
print("Saída:", OUTPUT_DIR.resolve())



## Universo de 60 ações

Foram definidos dez blocos setoriais com seis ações cada. O grupo fixo `pool_0` recebe a primeira ação de cada setor, `pool_1` recebe a segunda e assim sucessivamente.

Consequentemente, cada grupo de dez contém exatamente uma ação de cada setor, evitando que um grupo seja apenas tecnológico, apenas financeiro etc.


In [ ]:

SECTOR_TICKERS = {
    "information_technology": ["AAPL", "MSFT", "NVDA", "AVGO", "ORCL", "CRM"],
    "financials":             ["JPM", "BAC", "WFC", "GS", "MS", "AXP"],
    "health_care":            ["JNJ", "UNH", "LLY", "MRK", "ABBV", "TMO"],
    "consumer_discretionary": ["AMZN", "TSLA", "HD", "MCD", "NKE", "SBUX"],
    "consumer_staples":       ["WMT", "COST", "PG", "KO", "PEP", "PM"],
    "energy":                 ["XOM", "CVX", "COP", "SLB", "EOG", "MPC"],
    "industrials":            ["CAT", "DE", "BA", "GE", "UPS", "HON"],
    "communication_services": ["GOOGL", "META", "NFLX", "DIS", "VZ", "CMCSA"],
    "utilities":              ["NEE", "DUK", "SO", "AEP", "EXC", "SRE"],
    "materials_real_estate":  ["LIN", "APD", "SHW", "FCX", "AMT", "PLD"],
}

universe_rows = []
for sector, tickers in SECTOR_TICKERS.items():
    if len(tickers) != 6:
        raise ValueError(f"O setor {sector!r} não possui exatamente 6 tickers.")
    for pool_id, ticker in enumerate(tickers):
        universe_rows.append({
            "ticker": ticker,
            "sector": sector,
            "pool_id": int(pool_id),
            "pool_name": f"pool_{pool_id}",
        })

UNIVERSE_DF = (
    pd.DataFrame(universe_rows)
    .sort_values(["pool_id", "sector"])
    .reset_index(drop=True)
)

if UNIVERSE_DF["ticker"].nunique() != 60:
    raise ValueError("O universo precisa conter exatamente 60 tickers únicos.")

POOL_MAP = {
    int(pool_id): tuple(group["ticker"].tolist())
    for pool_id, group in UNIVERSE_DF.groupby("pool_id", sort=True)
}

display(UNIVERSE_DF.head(20))
print("\nTamanho dos grupos:")
display(
    UNIVERSE_DF.groupby("pool_id")
    .agg(n_tickers=("ticker", "size"), n_sectors=("sector", "nunique"))
)


In [ ]:
def save_table(df: pd.DataFrame, parquet_path: Path) -> Path:
    """Salva em Parquet; usa CSV como fallback quando necessário."""
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        csv_path = parquet_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        print(f"[aviso] Parquet indisponível ({exc!r}); salvo em {csv_path}.")
        return csv_path


def _extract_close_from_yfinance(
    raw: pd.DataFrame,
    requested_tickers: list[str],
) -> pd.DataFrame:
    """Extrai Close/Adj Close dos diferentes formatos do yfinance."""
    if raw is None or raw.empty:
        return pd.DataFrame()

    requested_tickers = [str(ticker).upper() for ticker in requested_tickers]

    if isinstance(raw.columns, pd.MultiIndex):
        level0 = set(map(str, raw.columns.get_level_values(0)))
        level1 = set(map(str, raw.columns.get_level_values(1)))

        for price_name in ["Close", "Adj Close"]:
            if price_name in level0:
                close = raw.xs(price_name, axis=1, level=0, drop_level=True).copy()
                if isinstance(close, pd.Series):
                    close = close.to_frame()
                close.columns = [str(column).upper() for column in close.columns]
                return close

        for price_name in ["Close", "Adj Close"]:
            if price_name in level1:
                close = raw.xs(price_name, axis=1, level=1, drop_level=True).copy()
                if isinstance(close, pd.Series):
                    close = close.to_frame()
                close.columns = [str(column).upper() for column in close.columns]
                return close

        return pd.DataFrame()

    for price_name in ["Close", "Adj Close"]:
        if price_name in raw.columns:
            close = raw[[price_name]].copy()
            close.columns = [requested_tickers[0]]
            return close

    return pd.DataFrame()


def _collect_yfinance_errors() -> dict[str, str]:
    shared = getattr(yf, "shared", None)
    errors = getattr(shared, "_ERRORS", {}) if shared is not None else {}
    return {
        str(ticker).upper(): str(message)
        for ticker, message in dict(errors or {}).items()
    }


def _download_chunk_with_retries(
    tickers: list[str],
    start: str,
    end: str,
    *,
    max_attempts: int = 4,
    timeout: float = 30.0,
) -> pd.DataFrame:
    """Baixa um lote pequeno, sem threads, com repetição e espera."""
    tickers = [str(ticker).upper() for ticker in tickers]

    for attempt in range(1, max_attempts + 1):
        try:
            raw = yf.download(
                tickers=tickers,
                start=start,
                end=end,
                interval="1d",
                auto_adjust=True,
                actions=False,
                group_by="column",
                threads=False,
                progress=False,
                repair=True,
                keepna=False,
                timeout=timeout,
                multi_level_index=True,
            )
            close = _extract_close_from_yfinance(raw, tickers)
            if not close.empty:
                close = close.sort_index()
                return close.loc[:, ~close.columns.duplicated()]

            print(
                f"[tentativa {attempt}/{max_attempts}] lote vazio: {tickers}"
            )
            errors = _collect_yfinance_errors()
            if errors:
                print("Erros reportados pelo yfinance:", errors)

        except Exception as exc:
            print(
                f"[tentativa {attempt}/{max_attempts}] {tickers}: "
                f"{type(exc).__name__}: {exc}"
            )

        if attempt < max_attempts:
            wait = min(45.0, 2.0 ** attempt + random.uniform(0.5, 2.0))
            print(f"Aguardando {wait:.1f} s...")
            time.sleep(wait)

    return pd.DataFrame()


def _download_single_ticker_fallback(
    ticker: str,
    start: str,
    end: str,
    *,
    max_attempts: int = 3,
) -> pd.Series | None:
    """Fallback individual usando Ticker.history."""
    ticker = str(ticker).upper()

    for attempt in range(1, max_attempts + 1):
        try:
            history = yf.Ticker(ticker).history(
                start=start,
                end=end,
                interval="1d",
                auto_adjust=True,
                actions=False,
                repair=True,
                raise_errors=False,
            )
            if history is not None and not history.empty:
                column = (
                    "Close" if "Close" in history.columns
                    else "Adj Close" if "Adj Close" in history.columns
                    else None
                )
                if column is not None:
                    series = history[column].copy()
                    series.name = ticker
                    if getattr(series.index, "tz", None) is not None:
                        series.index = series.index.tz_localize(None)
                    return series.sort_index()

            print(
                f"[fallback {ticker}] tentativa {attempt}/{max_attempts}: sem dados."
            )
        except Exception as exc:
            print(
                f"[fallback {ticker}] tentativa {attempt}/{max_attempts}: "
                f"{type(exc).__name__}: {exc}"
            )

        if attempt < max_attempts:
            time.sleep(min(30.0, 2.0 ** attempt + random.uniform(0.5, 1.5)))

    return None


def download_adjusted_close_robust(
    tickers: list[str],
    start: str,
    end: str,
    *,
    chunk_size: int = 8,
    max_attempts: int = 4,
    use_cache: bool = True,
) -> pd.DataFrame:
    """
    Download robusto:
    cache -> teste unitário -> lotes pequenos -> fallback individual.
    """
    tickers = [str(ticker).upper() for ticker in tickers]
    cache_path = RAW_DIR / "yfinance_adjusted_close_cache.csv"

    cached = pd.DataFrame()
    if use_cache and cache_path.exists():
        try:
            cached = pd.read_csv(cache_path, index_col=0, parse_dates=True)
            cached.columns = [str(column).upper() for column in cached.columns]
            cached = cached.loc[
                (cached.index >= pd.Timestamp(start))
                & (cached.index < pd.Timestamp(end))
            ]
            print("Cache recuperado:", cached.shape)
        except Exception as exc:
            print("[aviso] Falha ao ler cache:", exc)
            cached = pd.DataFrame()

    valid_cached = [
        ticker for ticker in tickers
        if ticker in cached.columns and cached[ticker].notna().any()
    ]
    pieces = [cached[valid_cached].copy()] if valid_cached else []

    missing = [ticker for ticker in tickers if ticker not in valid_cached]
    if not missing:
        return pd.concat(pieces, axis=1).reindex(columns=tickers).sort_index()

    diagnostic_ticker = missing[0]
    print(f"Teste de conexão com {diagnostic_ticker}...")
    diagnostic = _download_chunk_with_retries(
        [diagnostic_ticker],
        start,
        end,
        max_attempts=2,
    )
    if diagnostic.empty:
        raise RuntimeError(
            "O Yahoo Finance não retornou nem um ticker no teste inicial. "
            "Atualize o yfinance, reinicie o kernel e tente novamente. "
            "Também pode haver rate limit, bloqueio da rede, proxy ou SSL. "
            f"Erros internos: {_collect_yfinance_errors()}"
        )

    pieces.append(diagnostic)
    missing = [ticker for ticker in missing if ticker not in diagnostic.columns]

    n_chunks = math.ceil(len(missing) / chunk_size) if missing else 0
    for chunk_number, start_position in enumerate(
        range(0, len(missing), chunk_size),
        start=1,
    ):
        chunk = missing[start_position : start_position + chunk_size]
        print(f"\nLote {chunk_number}/{n_chunks}: {chunk}")

        close_chunk = _download_chunk_with_retries(
            chunk,
            start,
            end,
            max_attempts=max_attempts,
        )
        if not close_chunk.empty:
            pieces.append(close_chunk)

        partial = pd.concat(pieces, axis=1)
        partial = partial.loc[:, ~partial.columns.duplicated(keep="last")]
        partial.sort_index().to_csv(cache_path)
        time.sleep(random.uniform(1.0, 2.0))

    combined = pd.concat(pieces, axis=1)
    combined = combined.loc[:, ~combined.columns.duplicated(keep="last")]
    combined.columns = [str(column).upper() for column in combined.columns]
    combined = combined.sort_index()

    still_missing = [
        ticker for ticker in tickers
        if ticker not in combined.columns or not combined[ticker].notna().any()
    ]

    if still_missing:
        print("\nFallback individual:", still_missing)

    for ticker in still_missing:
        series = _download_single_ticker_fallback(ticker, start, end)
        if series is not None:
            combined[ticker] = series
            combined.sort_index().to_csv(cache_path)
        time.sleep(random.uniform(1.0, 2.0))

    combined = combined.loc[
        (combined.index >= pd.Timestamp(start))
        & (combined.index < pd.Timestamp(end))
    ].sort_index()
    combined.to_csv(cache_path)

    final_missing = [
        ticker for ticker in tickers
        if ticker not in combined.columns or not combined[ticker].notna().any()
    ]

    print("\nResumo do download")
    print("  solicitados:", len(tickers))
    print("  obtidos:", len(tickers) - len(final_missing))
    print("  ausentes:", final_missing)
    print("  dimensão:", combined.shape)
    print("  cache:", cache_path.resolve())

    if combined.empty:
        raise RuntimeError(
            "O Yahoo Finance permaneceu sem dados após todas as tentativas. "
            f"Erros internos: {_collect_yfinance_errors()}"
        )

    return combined


ALL_TICKERS = UNIVERSE_DF["ticker"].tolist()

PRICES_RAW = download_adjusted_close_robust(
    tickers=ALL_TICKERS,
    start=START_DATE,
    end=END_DATE,
    chunk_size=8,
    max_attempts=4,
    use_cache=True,
)

coverage = (
    PRICES_RAW.reindex(columns=ALL_TICKERS)
    .notna()
    .mean()
    .rename("coverage")
    .sort_values()
)
missing_download = sorted(
    ticker
    for ticker in ALL_TICKERS
    if ticker not in PRICES_RAW.columns
    or not PRICES_RAW[ticker].notna().any()
)

print("Dimensão bruta dos preços:", PRICES_RAW.shape)
print("Tickers ausentes:", missing_download)
display(coverage.head(15))

if missing_download:
    raise RuntimeError(
        "Ainda faltam tickers: "
        f"{missing_download}. O cache parcial foi preservado. "
        "Aguarde alguns minutos e execute novamente apenas esta célula."
    )

bad_coverage = coverage[coverage < MIN_PRICE_COVERAGE]
if not bad_coverage.empty:
    raise RuntimeError(
        "Há tickers com cobertura insuficiente:\n"
        f"{bad_coverage}"
    )

PRICES = (
    PRICES_RAW.reindex(columns=ALL_TICKERS)
    .ffill(limit=MAX_FORWARD_FILL_DAYS)
    .dropna(how="any")
)

RETURNS = (
    PRICES.pct_change(fill_method=None)
    .replace([np.inf, -np.inf], np.nan)
    .dropna(how="any")
)

if RETURNS.shape[0] < min(WINDOW_LENGTHS):
    raise RuntimeError(
        f"Há somente {RETURNS.shape[0]} retornos completos; "
        f"o mínimo esperado é {min(WINDOW_LENGTHS)}."
    )

save_table(
    PRICES.reset_index().rename(columns={PRICES.index.name or "index": "date"}),
    RAW_DIR / "adjusted_close_2023.parquet",
)
save_table(
    RETURNS.reset_index().rename(columns={RETURNS.index.name or "index": "date"}),
    RAW_DIR / "daily_returns_2023.parquet",
)
save_table(UNIVERSE_DF, TABLE_DIR / "universe_60.parquet")

print("Preços após limpeza:", PRICES.shape)
print("Retornos após limpeza:", RETURNS.shape)
display(RETURNS.describe().T.head())


## Construção física das instâncias

Para cada instância:

1. escolhe-se uma janela temporal;
2. realiza-se *moving-block bootstrap* para preservar parte da autocorrelação temporal;
3. calcula-se:
   \[
   \boldsymbol{\mu},\qquad \Sigma;
   \]
4. regulariza-se \(\Sigma\) com Ledoit–Wolf;
5. constrói-se o objetivo de seleção:
   \[
   f(\mathbf{x})
   =
   \gamma\,\mathbf{x}^{\mathsf T}\Sigma\mathbf{x}
   -(1-\gamma)\,\boldsymbol{\mu}^{\mathsf T}\mathbf{x},
   \qquad
   \sum_i x_i=K;
   \]
6. cria-se também o QUBO penalizado:
   \[
   f_{\mathrm{pen}}(\mathbf{x})
   =
   f(\mathbf{x})
   +A\left(\sum_i x_i-K\right)^2.
   \]

O banco guarda tanto o Hamiltoniano restrito ao subespaço de peso \(K\) quanto a versão penalizada.


In [ ]:

@dataclass(frozen=True)
class MarketStatistics:
    mu: np.ndarray
    cov: np.ndarray
    shrinkage: float
    window_start: str
    window_end: str
    window_length: int
    block_length: int
    covariance_condition_number: float


def moving_block_bootstrap(
    values: np.ndarray,
    sample_length: int,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Reamostragem em blocos móveis com reposição."""
    values = np.asarray(values, dtype=np.float64)
    n_rows = values.shape[0]

    if n_rows < 2:
        raise ValueError("A janela precisa conter ao menos duas observações.")

    block_length = int(max(1, min(block_length, n_rows)))
    n_blocks = int(math.ceil(sample_length / block_length))
    max_start = n_rows - block_length

    starts = rng.integers(0, max_start + 1, size=n_blocks)
    sampled_blocks = [
        values[start : start + block_length]
        for start in starts
    ]
    return np.concatenate(sampled_blocks, axis=0)[:sample_length]


def estimate_market_statistics(
    returns_df: pd.DataFrame,
    tickers: tuple[str, ...],
    rng: np.random.Generator,
) -> MarketStatistics:
    local = returns_df.loc[:, list(tickers)].dropna(how="any")
    max_length = len(local)

    valid_lengths = [
        length for length in WINDOW_LENGTHS
        if length <= max_length
    ]
    if not valid_lengths:
        raise ValueError(
            f"Não há janela válida para os tickers {tickers}."
        )

    window_length = int(rng.choice(valid_lengths))
    max_start = max_length - window_length
    start_position = int(rng.integers(0, max_start + 1))
    window = local.iloc[start_position : start_position + window_length]

    valid_blocks = [
        block for block in BLOCK_LENGTHS
        if block <= window_length
    ]
    block_length = int(rng.choice(valid_blocks))

    boot = moving_block_bootstrap(
        values=window.to_numpy(dtype=np.float64),
        sample_length=window_length,
        block_length=block_length,
        rng=rng,
    )

    mu = boot.mean(axis=0) * TRADING_DAYS

    covariance_model = LedoitWolf(assume_centered=False)
    covariance_model.fit(boot)
    cov = covariance_model.covariance_ * TRADING_DAYS
    cov = 0.5 * (cov + cov.T)

    eigenvalues = np.linalg.eigvalsh(cov)
    min_eig = float(eigenvalues.min())
    if min_eig < 0:
        cov = cov + (abs(min_eig) + 1e-10) * np.eye(cov.shape[0])

    condition_number = float(np.linalg.cond(cov))

    return MarketStatistics(
        mu=np.asarray(mu, dtype=np.float64),
        cov=np.asarray(cov, dtype=np.float64),
        shrinkage=float(covariance_model.shrinkage_),
        window_start=str(pd.Timestamp(window.index[0]).date()),
        window_end=str(pd.Timestamp(window.index[-1]).date()),
        window_length=window_length,
        block_length=block_length,
        covariance_condition_number=condition_number,
    )


def build_qubo_matrices(
    mu: np.ndarray,
    cov: np.ndarray,
    risk_aversion: float,
    k: int,
    penalty_multiplier: float,
) -> dict[str, Any]:
    """Retorna QUBO restrito e QUBO penalizado, ambos simétricos."""
    mu = np.asarray(mu, dtype=np.float64)
    cov = np.asarray(cov, dtype=np.float64)
    n = len(mu)

    q_base = (
        float(risk_aversion) * cov
        - (1.0 - float(risk_aversion)) * np.diag(mu)
    )
    q_base = 0.5 * (q_base + q_base.T)

    natural_scale = max(
        float(np.max(np.sum(np.abs(q_base), axis=1))),
        1e-8,
    )
    penalty_strength = float(penalty_multiplier) * natural_scale

    # A(1^T x - K)^2 = x^T[A 11^T - 2AK I]x + AK^2
    q_penalized = (
        q_base
        + penalty_strength * np.ones((n, n), dtype=np.float64)
        - 2.0 * penalty_strength * int(k) * np.eye(n)
    )
    penalty_constant = penalty_strength * (int(k) ** 2)

    return {
        "q_base": q_base,
        "q_penalized": q_penalized,
        "penalty_strength": penalty_strength,
        "penalty_constant": penalty_constant,
    }


def qubo_to_ising(
    q_matrix: np.ndarray,
    extra_constant: float = 0.0,
) -> tuple[np.ndarray, np.ndarray, float]:
    """Converte x^TQx + constante para offset + h.z + sum_{i<j}J_ij z_i z_j."""
    q = np.asarray(q_matrix, dtype=np.float64)
    q = 0.5 * (q + q.T)

    h = -0.5 * q.sum(axis=1)
    j_upper = 0.5 * np.triu(q, k=1)

    offset = (
        float(extra_constant)
        + 0.5 * float(np.trace(q))
        + 0.5 * float(np.triu(q, k=1).sum())
    )
    return h, j_upper, offset


def evaluate_ising(
    bitstring: np.ndarray,
    h: np.ndarray,
    j_upper: np.ndarray,
    offset: float,
) -> float:
    x = np.asarray(bitstring, dtype=np.float64)
    z = 1.0 - 2.0 * x
    pair_term = float(np.sum(j_upper * np.outer(z, z)))
    return float(offset + h @ z + pair_term)


def validate_qubo_ising_conversion() -> None:
    rng = np.random.default_rng(123)
    for n in [3, 5, 10]:
        a = rng.normal(size=(n, n))
        q = 0.5 * (a + a.T)
        extra = float(rng.normal())
        h, j, offset = qubo_to_ising(q, extra)

        for _ in range(20):
            x = rng.integers(0, 2, size=n)
            qubo_value = float(x @ q @ x + extra)
            ising_value = evaluate_ising(x, h, j, offset)
            if not np.isclose(qubo_value, ising_value, atol=1e-10):
                raise AssertionError(
                    f"Falha QUBO→Ising: {qubo_value=} != {ising_value=}"
                )


validate_qubo_ising_conversion()
print("Conversão QUBO → Ising validada numericamente.")


In [ ]:

def deterministic_id(*parts: Any) -> str:
    payload = "|".join(map(str, parts)).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:20]


def stratified_subset(
    universe_df: pd.DataFrame,
    n_assets: int,
    rng: np.random.Generator,
) -> tuple[str, ...]:
    """Seleciona ao menos uma ação de cada um dos dez setores."""
    sectors = sorted(universe_df["sector"].unique())
    if n_assets < len(sectors):
        raise ValueError(
            f"n_assets={n_assets} é menor que os {len(sectors)} setores."
        )

    selected: list[str] = []
    for sector in sectors:
        sector_tickers = universe_df.loc[
            universe_df["sector"] == sector,
            "ticker",
        ].to_numpy()
        selected.append(str(rng.choice(sector_tickers)))

    remaining = universe_df.loc[
        ~universe_df["ticker"].isin(selected),
        "ticker",
    ].to_numpy()

    n_extra = int(n_assets - len(selected))
    if n_extra > 0:
        selected.extend(
            map(str, rng.choice(remaining, size=n_extra, replace=False))
        )

    rng.shuffle(selected)
    return tuple(selected)


def sample_risk_aversion(rng: np.random.Generator) -> float:
    # Beta(2,2) evita concentração excessiva nos extremos.
    beta_value = float(rng.beta(2.0, 2.0))
    return (
        RISK_AVERSION_MIN
        + beta_value * (RISK_AVERSION_MAX - RISK_AVERSION_MIN)
    )


def padded_vector(values: np.ndarray, max_n: int = MAX_N) -> np.ndarray:
    out = np.zeros(max_n, dtype=np.float32)
    out[: len(values)] = np.asarray(values, dtype=np.float32)
    return out


def padded_matrix(values: np.ndarray, max_n: int = MAX_N) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    out = np.zeros((max_n, max_n), dtype=np.float32)
    n = values.shape[0]
    out[:n, :n] = values
    return out


def padded_tickers(tickers: tuple[str, ...], max_n: int = MAX_N) -> np.ndarray:
    out = np.full(max_n, "", dtype="<U8")
    out[: len(tickers)] = list(tickers)
    return out


TOTAL = EXPECTED_TOTAL

arrays: dict[str, np.ndarray] = {
    "n_assets": np.zeros(TOTAL, dtype=np.int16),
    "k_selected": np.zeros(TOTAL, dtype=np.int16),
    "asset_mask": np.zeros((TOTAL, MAX_N), dtype=np.uint8),
    "tickers": np.full((TOTAL, MAX_N), "", dtype="<U8"),
    "mu": np.zeros((TOTAL, MAX_N), dtype=np.float32),
    "cov": np.zeros((TOTAL, MAX_N, MAX_N), dtype=np.float32),
    "qubo_base": np.zeros((TOTAL, MAX_N, MAX_N), dtype=np.float32),
    "qubo_penalized": np.zeros((TOTAL, MAX_N, MAX_N), dtype=np.float32),
    "ising_h_base": np.zeros((TOTAL, MAX_N), dtype=np.float32),
    "ising_J_base": np.zeros((TOTAL, MAX_N, MAX_N), dtype=np.float32),
    "ising_offset_base": np.zeros(TOTAL, dtype=np.float64),
    "ising_h_penalized": np.zeros((TOTAL, MAX_N), dtype=np.float32),
    "ising_J_penalized": np.zeros((TOTAL, MAX_N, MAX_N), dtype=np.float32),
    "ising_offset_penalized": np.zeros(TOTAL, dtype=np.float64),
    "optimal_bitstring": np.zeros((TOTAL, MAX_N), dtype=np.uint8),
    "exact_objective": np.full(TOTAL, np.nan, dtype=np.float64),
    "exact_expected_return": np.full(TOTAL, np.nan, dtype=np.float64),
    "exact_risk": np.full(TOTAL, np.nan, dtype=np.float64),
    "exact_degeneracy": np.zeros(TOTAL, dtype=np.int32),
}

metadata_rows: list[dict[str, Any]] = []
cursor = 0


def append_instance(
    *,
    tickers: tuple[str, ...],
    stats: MarketStatistics,
    n_assets: int,
    k_selected: int,
    risk_aversion: float,
    base_market_id: str,
    source_family: str,
    pool_id: int | None,
    generation_seed: int,
) -> None:
    global cursor

    if cursor >= TOTAL:
        raise IndexError("Número de instâncias excedeu o total pré-alocado.")

    qubos = build_qubo_matrices(
        mu=stats.mu,
        cov=stats.cov,
        risk_aversion=risk_aversion,
        k=k_selected,
        penalty_multiplier=PENALTY_MULTIPLIER,
    )

    h_base, j_base, offset_base = qubo_to_ising(
        qubos["q_base"],
        extra_constant=0.0,
    )
    h_pen, j_pen, offset_pen = qubo_to_ising(
        qubos["q_penalized"],
        extra_constant=qubos["penalty_constant"],
    )

    config_id = f"N{n_assets}_K{k_selected}"
    instance_id = deterministic_id(
        config_id,
        base_market_id,
        generation_seed,
        ",".join(tickers),
    )

    arrays["n_assets"][cursor] = n_assets
    arrays["k_selected"][cursor] = k_selected
    arrays["asset_mask"][cursor, :n_assets] = 1
    arrays["tickers"][cursor] = padded_tickers(tickers)
    arrays["mu"][cursor] = padded_vector(stats.mu)
    arrays["cov"][cursor] = padded_matrix(stats.cov)
    arrays["qubo_base"][cursor] = padded_matrix(qubos["q_base"])
    arrays["qubo_penalized"][cursor] = padded_matrix(qubos["q_penalized"])
    arrays["ising_h_base"][cursor] = padded_vector(h_base)
    arrays["ising_J_base"][cursor] = padded_matrix(j_base)
    arrays["ising_offset_base"][cursor] = offset_base
    arrays["ising_h_penalized"][cursor] = padded_vector(h_pen)
    arrays["ising_J_penalized"][cursor] = padded_matrix(j_pen)
    arrays["ising_offset_penalized"][cursor] = offset_pen

    metadata_rows.append({
        "row_index": int(cursor),
        "instance_id": instance_id,
        "config_id": config_id,
        "source_family": source_family,
        "paired_market_id": base_market_id,
        "pool_id": pool_id,
        "n_assets": int(n_assets),
        "k_selected": int(k_selected),
        "tickers_json": json.dumps(list(tickers)),
        "generation_seed": int(generation_seed),
        "risk_aversion": float(risk_aversion),
        "penalty_strength": float(qubos["penalty_strength"]),
        "penalty_constant": float(qubos["penalty_constant"]),
        "window_start": stats.window_start,
        "window_end": stats.window_end,
        "window_length": int(stats.window_length),
        "block_length": int(stats.block_length),
        "covariance_shrinkage": float(stats.shrinkage),
        "covariance_condition_number": float(
            stats.covariance_condition_number
        ),
        "exact_status": "pending",
        "qfim_status": "not_computed",
        "vqe_status": "not_computed",
    })
    cursor += 1


master_rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# Família A: N=10 pareado em K=2,...,7
# Cada base de mercado é reutilizada nos seis valores de K.
# ------------------------------------------------------------
for base_index in tqdm(
    range(N_INSTANCES_PER_CONFIG),
    desc="Gerando bases N=10 pareadas",
):
    pool_id = int(base_index % 6)
    tickers = POOL_MAP[pool_id]

    seed = int(master_rng.integers(0, 2**31 - 1))
    local_rng = np.random.default_rng(seed)

    stats = estimate_market_statistics(
        returns_df=RETURNS,
        tickers=tickers,
        rng=local_rng,
    )
    risk_aversion = sample_risk_aversion(local_rng)
    base_market_id = f"N10_BASE_{base_index:06d}_POOL_{pool_id}"

    for k_selected in N10_K_VALUES:
        append_instance(
            tickers=tickers,
            stats=stats,
            n_assets=10,
            k_selected=k_selected,
            risk_aversion=risk_aversion,
            base_market_id=base_market_id,
            source_family="fixed_diversified_pool_10",
            pool_id=pool_id,
            generation_seed=seed,
        )

# ------------------------------------------------------------
# Família B: mudanças de N, mantendo K=5.
# ------------------------------------------------------------
for n_assets, k_selected in EXTRA_CONFIGS:
    for base_index in tqdm(
        range(N_INSTANCES_PER_CONFIG),
        desc=f"Gerando N={n_assets}, K={k_selected}",
    ):
        seed = int(master_rng.integers(0, 2**31 - 1))
        local_rng = np.random.default_rng(seed)

        tickers = stratified_subset(
            universe_df=UNIVERSE_DF,
            n_assets=n_assets,
            rng=local_rng,
        )
        stats = estimate_market_statistics(
            returns_df=RETURNS,
            tickers=tickers,
            rng=local_rng,
        )
        risk_aversion = sample_risk_aversion(local_rng)
        base_market_id = f"N{n_assets}_BASE_{base_index:06d}"

        append_instance(
            tickers=tickers,
            stats=stats,
            n_assets=n_assets,
            k_selected=k_selected,
            risk_aversion=risk_aversion,
            base_market_id=base_market_id,
            source_family="stratified_subset_60",
            pool_id=None,
            generation_seed=seed,
        )

if cursor != TOTAL:
    raise RuntimeError(f"Foram preenchidas {cursor} linhas; esperadas {TOTAL}.")

METADATA = pd.DataFrame(metadata_rows)

print("Banco construído:", METADATA.shape)
display(METADATA.groupby(["config_id", "source_family"]).size().rename("n"))
display(METADATA.head())



## Solução exata clássica

Como \(N\leq20\), é possível enumerar apenas os bitstrings de peso \(K\):

\[
\binom{N}{K}.
\]

O maior caso utilizado é:

\[
\binom{20}{5}=15.504,
\]

o que ainda permite produzir rótulos exatos para o banco.

A solução exata é calculada sobre o objetivo restrito `qubo_base`. A penalização permanece disponível para o Hamiltoniano quântico, mas não é necessária para a enumeração porque todos os candidatos já satisfazem \(\sum_i x_i=K\).


In [ ]:

def candidate_matrix(n_assets: int, k_selected: int) -> np.ndarray:
    n_candidates = math.comb(n_assets, k_selected)
    candidates = np.zeros(
        (n_candidates, n_assets),
        dtype=np.uint8,
    )
    for row, indices in enumerate(
        combinations(range(n_assets), k_selected)
    ):
        candidates[row, list(indices)] = 1
    return candidates


def solve_exact_group(
    row_indices: np.ndarray,
    n_assets: int,
    k_selected: int,
    batch_size: int,
) -> None:
    candidates_u8 = candidate_matrix(n_assets, k_selected)
    candidates = candidates_u8.astype(np.float64)

    for start in tqdm(
        range(0, len(row_indices), batch_size),
        desc=f"Exato N={n_assets}, K={k_selected}",
        leave=False,
    ):
        block_indices = row_indices[start : start + batch_size]
        q_batch = arrays["qubo_base"][
            block_indices,
            :n_assets,
            :n_assets,
        ].astype(np.float64)

        # scores[b,c] = x_c^T Q_b x_c
        scores = np.einsum(
            "ci,bij,cj->bc",
            candidates,
            q_batch,
            candidates,
            optimize=True,
        )

        best_positions = np.argmin(scores, axis=1)
        best_values = scores[
            np.arange(len(block_indices)),
            best_positions,
        ]

        for local_position, row_index in enumerate(block_indices):
            best_bits = candidates_u8[best_positions[local_position]]
            arrays["optimal_bitstring"][
                row_index,
                :n_assets,
            ] = best_bits
            arrays["exact_objective"][row_index] = float(
                best_values[local_position]
            )

            mu = arrays["mu"][row_index, :n_assets].astype(np.float64)
            cov = arrays["cov"][
                row_index,
                :n_assets,
                :n_assets,
            ].astype(np.float64)

            arrays["exact_expected_return"][row_index] = float(
                best_bits @ mu
            )
            arrays["exact_risk"][row_index] = float(
                best_bits @ cov @ best_bits
            )

            arrays["exact_degeneracy"][row_index] = int(
                np.isclose(
                    scores[local_position],
                    best_values[local_position],
                    atol=1e-10,
                    rtol=1e-8,
                ).sum()
            )


if COMPUTE_EXACT_LABELS:
    for config_id, group in METADATA.groupby("config_id", sort=True):
        n_assets = int(group["n_assets"].iloc[0])
        k_selected = int(group["k_selected"].iloc[0])
        row_indices = group["row_index"].to_numpy(dtype=int)

        if n_assets == 20:
            batch_size = EXACT_SOLVER_BATCH_SIZE_N20
        elif n_assets >= 13:
            batch_size = 32
        else:
            batch_size = 128

        solve_exact_group(
            row_indices=row_indices,
            n_assets=n_assets,
            k_selected=k_selected,
            batch_size=batch_size,
        )

    METADATA["exact_status"] = "computed"
    METADATA["exact_objective"] = arrays["exact_objective"]
    METADATA["exact_expected_return"] = arrays["exact_expected_return"]
    METADATA["exact_risk"] = arrays["exact_risk"]
    METADATA["exact_degeneracy"] = arrays["exact_degeneracy"]
    METADATA["optimal_bitstring"] = [
        "".join(
            map(
                str,
                arrays["optimal_bitstring"][row, : int(n_assets)].tolist(),
            )
        )
        for row, n_assets in enumerate(METADATA["n_assets"])
    ]
else:
    print("COMPUTE_EXACT_LABELS=False: rótulos exatos não foram calculados.")

display(
    METADATA.groupby("config_id")
    .agg(
        n=("instance_id", "size"),
        exact_objective_median=("exact_objective", "median"),
        exact_return_median=("exact_expected_return", "median"),
        exact_risk_median=("exact_risk", "median"),
        degeneracy_median=("exact_degeneracy", "median"),
    )
)



## Divisões experimentais sem vazamento

O banco recebe diferentes rótulos de divisão:

- `split_iid`: divisão aleatória agrupada por `paired_market_id`, garantindo que versões com \(K\) diferentes da mesma base de mercado permaneçam juntas;
- `split_size_ood`: treina em \(N=10\), valida em \(N=13\) e testa em \(N=20\);
- `split_k_ood`: para \(N=10\), separa valores de \(K\) vistos e não vistos;
- `split_pool_ood`: mantém um grupo fixo de dez ações para validação e outro para teste.

Essas divisões serão úteis posteriormente para avaliar se o Transformer apenas memoriza Hamiltonianos ou realmente generaliza.


In [ ]:

def assign_grouped_iid_split(
    metadata: pd.DataFrame,
    seed: int,
) -> pd.Series:
    rng = np.random.default_rng(seed)
    unique_groups = metadata["paired_market_id"].drop_duplicates().to_numpy()
    rng.shuffle(unique_groups)

    n_groups = len(unique_groups)
    n_train = int(round(0.70 * n_groups))
    n_val = int(round(0.15 * n_groups))

    train_groups = set(unique_groups[:n_train])
    val_groups = set(unique_groups[n_train : n_train + n_val])

    return metadata["paired_market_id"].map(
        lambda group: (
            "train" if group in train_groups
            else "validation" if group in val_groups
            else "test"
        )
    )


METADATA["split_iid"] = assign_grouped_iid_split(
    METADATA,
    seed=RANDOM_SEED + 1,
)

METADATA["split_size_ood"] = np.select(
    [
        METADATA["n_assets"].eq(10),
        METADATA["n_assets"].eq(13),
        METADATA["n_assets"].eq(20),
    ],
    ["train", "validation", "test"],
    default="unused",
)

METADATA["split_k_ood"] = "unused"
n10_mask = METADATA["n_assets"].eq(10)
METADATA.loc[
    n10_mask & METADATA["k_selected"].isin([2, 4, 6]),
    "split_k_ood",
] = "train"
METADATA.loc[
    n10_mask & METADATA["k_selected"].isin([3, 5]),
    "split_k_ood",
] = "validation"
METADATA.loc[
    n10_mask & METADATA["k_selected"].eq(7),
    "split_k_ood",
] = "test"

METADATA["split_pool_ood"] = "train"
METADATA.loc[
    METADATA["pool_id"].eq(4),
    "split_pool_ood",
] = "validation"
METADATA.loc[
    METADATA["pool_id"].eq(5),
    "split_pool_ood",
] = "test"
METADATA.loc[
    METADATA["pool_id"].isna(),
    "split_pool_ood",
] = "unused"

print("Split IID:")
display(METADATA.groupby(["config_id", "split_iid"]).size().unstack(fill_value=0))

print("\nSplit OOD por tamanho:")
display(METADATA.groupby(["config_id", "split_size_ood"]).size().unstack(fill_value=0))


In [ ]:

def run_quality_audit(
    metadata: pd.DataFrame,
    arrays_dict: dict[str, np.ndarray],
    n_random_checks: int = 200,
) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_SEED + 99)
    rows = []

    rows.append({
        "check": "total_instances",
        "value": int(len(metadata)),
        "expected": int(EXPECTED_TOTAL),
        "passed": bool(len(metadata) == EXPECTED_TOTAL),
    })

    counts = metadata.groupby("config_id").size()
    rows.append({
        "check": "minimum_per_configuration",
        "value": int(counts.min()),
        "expected": int(N_INSTANCES_PER_CONFIG),
        "passed": bool((counts == N_INSTANCES_PER_CONFIG).all()),
    })

    sampled_rows = rng.choice(
        len(metadata),
        size=min(n_random_checks, len(metadata)),
        replace=False,
    )

    psd_ok = True
    cardinality_ok = True
    ising_ok = True

    for row_index in sampled_rows:
        n_assets = int(arrays_dict["n_assets"][row_index])
        k_selected = int(arrays_dict["k_selected"][row_index])

        cov = arrays_dict["cov"][
            row_index,
            :n_assets,
            :n_assets,
        ].astype(np.float64)

        if np.linalg.eigvalsh(cov).min() < -1e-7:
            psd_ok = False

        if COMPUTE_EXACT_LABELS:
            bits = arrays_dict["optimal_bitstring"][
                row_index,
                :n_assets,
            ].astype(np.float64)
            if int(bits.sum()) != k_selected:
                cardinality_ok = False

        random_bits = rng.integers(0, 2, size=n_assets)
        q_pen = arrays_dict["qubo_penalized"][
            row_index,
            :n_assets,
            :n_assets,
        ].astype(np.float64)

        qubo_value = float(
            random_bits @ q_pen @ random_bits
            + metadata.iloc[row_index]["penalty_constant"]
        )
        ising_value = evaluate_ising(
            bitstring=random_bits,
            h=arrays_dict["ising_h_penalized"][
                row_index,
                :n_assets,
            ],
            j_upper=arrays_dict["ising_J_penalized"][
                row_index,
                :n_assets,
                :n_assets,
            ],
            offset=float(
                arrays_dict["ising_offset_penalized"][row_index]
            ),
        )
        if not np.isclose(qubo_value, ising_value, atol=2e-5):
            ising_ok = False

    rows.extend([
        {
            "check": "covariance_psd_random_sample",
            "value": psd_ok,
            "expected": True,
            "passed": psd_ok,
        },
        {
            "check": "exact_cardinality_random_sample",
            "value": cardinality_ok,
            "expected": True,
            "passed": cardinality_ok,
        },
        {
            "check": "penalized_qubo_ising_equivalence",
            "value": ising_ok,
            "expected": True,
            "passed": ising_ok,
        },
    ])

    return pd.DataFrame(rows)


QUALITY_AUDIT = run_quality_audit(METADATA, arrays)
display(QUALITY_AUDIT)

if not QUALITY_AUDIT["passed"].all():
    raise AssertionError("A auditoria encontrou uma ou mais falhas.")



## Salvamento do banco

O banco é salvo em duas formas complementares:

### Metadados tabulares

`problem_instances_metadata.parquet`

Contém identificadores, configuração \(N,K\), ações, janela, sementes, parâmetros de geração, splits e solução exata.

### Tensores para modelos

`problem_instances_model_ready.npz`

Todos os tensores são preenchidos até `MAX_N=20`, acompanhados por `asset_mask`. Isso permite treinar um único modelo com \(N=10,13,20\).

As matrizes armazenadas incluem:

- \(\mu\);
- \(\Sigma\);
- QUBO restrito;
- QUBO penalizado;
- \(h\), \(J\) e *offset* de Ising;
- bitstring exato;
- máscaras de ativos.

Também são criadas tabelas vazias para os rótulos futuros de VQE, \(\theta\) e QFIM.


In [ ]:

metadata_path = save_table(
    METADATA,
    TABLE_DIR / "problem_instances_metadata.parquet",
)
audit_path = save_table(
    QUALITY_AUDIT,
    REPORT_DIR / "quality_audit.parquet",
)

npz_path = ARRAY_DIR / "problem_instances_model_ready.npz"
np.savez_compressed(npz_path, **arrays)

configuration_counts = (
    METADATA.groupby(["config_id", "source_family"])
    .size()
    .rename("n_instances")
    .reset_index()
)
counts_path = save_table(
    configuration_counts,
    REPORT_DIR / "configuration_counts.parquet",
)

# Tabela vazia para enriquecimento futuro com VQE e QFIM.
FUTURE_LABEL_COLUMNS = [
    "instance_id",
    "solution_id",
    "theta_initial_json",
    "theta_optimal_json",
    "theta_periods_json",
    "vqe_best_energy",
    "vqe_gap",
    "vqe_probability_best",
    "vqe_nfev",
    "vqe_optimizer_time",
    "qfim_evaluation_point_json",
    "qfim_rank",
    "qfim_diagonal_json",
    "qfim_eigenvalues_json",
    "qfim_top_indices_json",
    "qfim_local_relevance_json",
    "qfim_status",
    "vqe_status",
]
future_labels_path = save_table(
    pd.DataFrame(columns=FUTURE_LABEL_COLUMNS),
    LABEL_DIR / "vqe_theta_qfim_labels_empty.parquet",
)

manifest = {
    "name": "diversidade_dos_dados_yfinance_2023_2024",
    "version": "1.0.0",
    "created_by_notebook": "00_DIVERSIDADE_DOS_DADOS_YFINANCE_2023_2024.ipynb",
    "date_interval": {
        "start_inclusive": START_DATE,
        "end_exclusive": END_DATE,
    },
    "universe_size": int(UNIVERSE_DF["ticker"].nunique()),
    "max_n": int(MAX_N),
    "instances_per_configuration": int(N_INSTANCES_PER_CONFIG),
    "expected_total_instances": int(EXPECTED_TOTAL),
    "actual_total_instances": int(len(METADATA)),
    "configurations": configuration_counts.to_dict(orient="records"),
    "n10_k_values": N10_K_VALUES,
    "extra_configurations": [
        {"n_assets": int(n), "k_selected": int(k)}
        for n, k in EXTRA_CONFIGS
    ],
    "qfim_computed": False,
    "vqe_theta_computed": False,
    "exact_labels_computed": bool(COMPUTE_EXACT_LABELS),
    "array_file": str(npz_path),
    "metadata_file": str(metadata_path),
    "quality_audit_file": str(audit_path),
    "future_labels_file": str(future_labels_path),
    "array_schema": {
        key: {
            "shape": list(value.shape),
            "dtype": str(value.dtype),
        }
        for key, value in arrays.items()
    },
}

manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Arquivos salvos:")
for path in [
    metadata_path,
    npz_path,
    counts_path,
    audit_path,
    future_labels_path,
    manifest_path,
]:
    print(" -", Path(path).resolve())


In [ ]:

# ============================================================
# CONFERÊNCIA FINAL
# ============================================================

print("Resumo por configuração:")
display(
    METADATA.groupby(["config_id", "n_assets", "k_selected"])
    .agg(
        n_instances=("instance_id", "size"),
        n_market_bases=("paired_market_id", "nunique"),
        n_unique_asset_sets=("tickers_json", "nunique"),
        risk_aversion_min=("risk_aversion", "min"),
        risk_aversion_median=("risk_aversion", "median"),
        risk_aversion_max=("risk_aversion", "max"),
        exact_gap_available=("exact_objective", "count"),
    )
    .reset_index()
)

print("\nExemplo pareado: mesmos dados de mercado, K diferente")
example_base = METADATA.loc[
    METADATA["n_assets"].eq(10),
    "paired_market_id",
].iloc[0]

display(
    METADATA.loc[
        METADATA["paired_market_id"].eq(example_base),
        [
            "instance_id",
            "paired_market_id",
            "pool_id",
            "n_assets",
            "k_selected",
            "risk_aversion",
            "window_start",
            "window_end",
            "optimal_bitstring",
            "exact_objective",
        ],
    ].sort_values("k_selected")
)



# Próxima etapa

Depois de validar este banco, o passo seguinte será enriquecê-lo com:

1. múltiplos restarts do VQE;
2. \(\theta\) inicial e \(\theta^*\);
3. energia, probabilidade, bitstring e `nfev`;
4. QFIM em pontos definidos da trajetória;
5. rank, diagonal, autovalores e relevância local;
6. comparação dos índices ativos quando mudam:
   - os ativos financeiros;
   - \(K\);
   - \(N\);
   - o Hamiltoniano.

A primeira pergunta científica será:

\[
\boxed{
\text{Os mesmos sete índices permanecem relevantes em diferentes }
(N,K)\text{ e diferentes universos?}
}
\]

Somente depois dessa análise será seguro decidir entre:

- duas cabeças fixas \(7+23\);
- uma cabeça dinâmica de relevância;
- ou uma arquitetura híbrida.
